# Quantification of shape during growth of propagules

## 1. Documentation and Imports


Created on 01-04-2021

Author: Valentin Laplaud

This code is performing measurements on the propagule contours computed by 'ShapeComputationNB'

In [ ]:
# plotting stuff
import matplotlib as mpl
mpl.use('TkAgg')
%matplotlib inline
import matplotlib.pyplot as plt
from cycler import cycler
#Default colors
colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/30) for i in range(30)]
mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)

# numbers handling
import numpy as np
import pandas as pd

# signal processing 
from scipy.signal import savgol_filter, correlate, correlation_lags
from scipy.interpolate import interp1d

# images handling
from skimage import io
from skimage.filters import threshold_otsu, gaussian, laplace, sobel
from skimage.measure import label, regionprops, regionprops_table
from skimage.util import invert
from skimage.morphology import binary_opening, binary_closing, remove_small_holes,binary_erosion
from skimage.color import rgb2gray
from skimage.segmentation import active_contour, morphological_geodesic_active_contour,morphological_chan_vese, checkerboard_level_set, inverse_gaussian_gradient
import cv2 as cv

# to hide known warnings
import warnings

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../')
import VallapFunc as vf


## 2. Define analysis functions

### 2.1 Mean contour over all propagule

Computation only for the number of images of the shortest video. Ponctually one gemma can be ignored for one time point if the focus on the video is bad.

In [ ]:
def computeMeanContour(StackList,ContourData,GlobalData):
    
    # Number of image in the shortest videos
    nmin = 1 + int(np.min([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    
    Theta = ContourData[ContourData['Img']==0].loc[StackList[0],'Theta'].to_numpy()
    
    # Storing variables
    alignedRs = np.empty((len(Theta),len(StackList)))
    alignedRs[:] = np.nan
    
    for i in range(nmin):
        
        print('Computing mean contour for image : ' + str(i) + '/' + str(nmin).ljust(5), end='\r')
        
        # On each image, retrieve and align contour for each propagule
        
        for ii,s in zip(range(len(StackList)),StackList):
           
            rn = ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == i),'alignR']            
            
            if bool(GlobalData.loc[(GlobalData.index == s) &
                                   (GlobalData['Img'] == i),'GoodFocus'].to_numpy) == True:
                alignedRs[:,ii] = rn

        # Computing mean of aligned contours        
        meanR = np.nanmean(alignedRs,1) 
        
        meanX,meanY = vf.ToCart(Theta, meanR, angle='deg')
        
        # Storing contour data
        data = {'X':meanX,
                    'Y':meanY,  
                    'alignR':meanR,  
                    'Theta':Theta,
                    'Img':i*np.ones(len(meanX))} 
      
        newdata = pd.DataFrame(data=data,index = np.repeat('meanCont',len(meanX)))
        
        ContourData = ContourData.append(newdata) # adding to global dataframe

    print('Computing mean contour or image : ' + str(nmin) + '/' + str(nmin).ljust(5), end='\n\n')
        
    return(ContourData)

### 2.2 Computation of growth and similarity 

Similarity : 
Comparison of one normalized contour in time rn(theta,t) with either the first contour of the same gemma rn0(theta,t) - selfSimilarity- or with the average contour r(theta,t) -meanSimilarity-.

Growth : 
Percentage of radial extension Gn(theta, t)

In [ ]:
def computeGrowthAndSimi(StackList,ContourData,GlobalData, **kwargs):  
       
    # define storing variables
    ContourData['Growth'] = np.nan    
    GlobalData['meanSimi'] = np.nan
    GlobalData['selfSimi'] = np.nan    
       
    # Number of image in the shortest and longest videos
    tmax = 1 + int(np.max([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    tmin = 1 + int(np.min([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    
    
    Theta = ContourData.loc[(ContourData.index == StackList[0]) & (ContourData['Img'] == 0),'Theta']
        
    GrowthMat = np.empty((len(StackList),len(Theta),tmax)) # Growth Gn(Theta,t) 
        
    for ni,n in zip(range(len(StackList)),StackList):
        
        print('Processing ' + n + ' :')
            
        timg = int(1 + np.max(ContourData.loc[n, 'Img'])) # number of images
        
        for ti in range(timg):
            
            print('Computing similarities and growth for image : ' + str(ti+1) + '/' + str(timg).ljust(5), end='\r')
            
            # Normalized radius and start (t=0) radius
            Rn = ContourData.loc[(ContourData.index == n) & (ContourData['Img'] == ti),'alignRnorm'].to_numpy()
            Rn0 = ContourData.loc[(ContourData.index == n) & (ContourData['Img'] == 0),'alignRnorm'].to_numpy()
            
            # Computing self similarity
            selfSimi = np.sqrt(np.mean(np.square(Rn-Rn0)))
            GlobalData.loc[(GlobalData.index == n) & (GlobalData['Img'] == ti),'selfSimi'] = selfSimi
            
            # Radius, start raidus, and mean radius
            rn = ContourData.loc[(ContourData.index == n) & (ContourData['Img'] == ti),'alignR'].to_numpy()
            rn0 = ContourData.loc[(ContourData.index == n) & (ContourData['Img'] == 0),'alignR'].to_numpy()
            r = ContourData.loc[(ContourData.index == 'meanCont') & (ContourData['Img'] == ti), 'alignR'].to_numpy()
            
            if not ti >= tmin:
                # Computing mean similarity
                meanSimi = np.sqrt(np.mean(np.square(rn-r))) 

                GlobalData.loc[(GlobalData.index == n) & (GlobalData['Img'] == ti),'meanSimi'] = meanSimi
            
                
            # Growth
            Gn = (np.divide(rn,rn0)-1)*100
            
            ContourData.loc[(ContourData.index == n) & (ContourData['Img'] == ti),'Growth'] = Gn
            
            GrowthMat[ni,:,ti] = Gn
        
        print('Computing similarities and growth for image : ' + str(ti+1) + '/' + str(timg).ljust(5), end='\n\n')    
            
    return(ContourData,GlobalData,GrowthMat)         
            

### 2.3 Estimating symmetry of the shape

#### 2.3.0 Symmetry computation

Computes the symmetry of a radially define contour, with an axis of symmetry going through Theta0 -> Theta0+180

In [ ]:
def Sym(Theta,R,Theta0):
    
    npts = len(Theta)
    i0 = np.argmin(np.absolute(Theta-Theta0))+1
    i180 = int(i0+npts/2)
    
    R1 = np.roll(R,-i0)[0:int(npts/2-1)]
    R2 = np.flip(np.roll(R,-i180)[0:int(npts/2-1)])

    aSym = np.sqrt(np.mean(np.square(R1-R2)))
    
    return(aSym)

#### 2.3.1 Compute symmetry for all contours

In [ ]:
def computeSym(StackList,ContourData,GlobalData):    
      
    GlobalData['Assymmetry'] = np.nan
    
    for n in StackList:
        
        timg = int(1 + np.max(ContourData.loc[n, 'Img'])) # number of images
        
        for ti in range(timg):
            
            Theta = ContourData.loc[(ContourData.index == n) & (ContourData['Img'] == ti),'Theta'].to_numpy()
            rn = ContourData.loc[(ContourData.index == n) & (ContourData['Img'] == ti),'alignR'].to_numpy()
            
            aSym = Sym(Theta,rn,0)
            
            GlobalData.loc[(GlobalData.index == n) & (GlobalData['Img'] == ti),'Assymmetry'] = aSym
            
    return(ContourData,GlobalData)

## 3. Plot fuctions

### 3.1 Plot propagules area in time

In [ ]:
def plotArea(StackList, Data, P, Title, **kwargs):

    # Colors
    colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/30) for i in range(30)]
    mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)
    
    #init and read kwargs
    Trap = False # default value
    soloGraphs = False
    
    for key, value in kwargs.items(): 
        if key == 'Trapped':
            Trap = value
        elif key == 'solo':
            soloGraphs = value
        else:
            print('Unknown key : ' + key+ '. Kwarg ignored.')
         
    # check existence of figure folder, if absent, create it
    if not os.path.exists(P + '\\Figures'):
            os.mkdir(P + '\\Figures') # create folder
    
    # plot figure
    
    ###################### plot individual graphs #######################
    
    if soloGraphs: 
    
        for s in StackList:
            

            fig1 = plt.figure(dpi = 200)
            fig1.suptitle(s + ' - Size vs. time')
            plt.xlabel('Time (min)')
            plt.ylabel('Size (mm²)')
            

            xTime = Data.at[s, 'Time (min)'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        
            
            PropBBOX = Data.at[s, 'Bounding box (px)']

            plt.plot(xTime,Data.at[s, 'Size (mm²)'].to_numpy()[Data.at[s, 'GoodFocus']==True],'-o',ms=3,figure=fig1,label=s)

            # if there is a trap, find out when th ppgls touches up and bottom wall
            if Trap:
                trapdata =vf.LoadImageJResults(P + '\\' + s + '_Trap.txt', ['BX','BY','Width','Height'])
                TrapBBOX = [trapdata.at[0, 'BY'], trapdata.at[0, 'BX'],trapdata.at[0, 'BY'] + trapdata.at[0, 'Height'] ,
                        trapdata.at[0, 'BX'] + trapdata.at[0, 'Width']]

                BBOXDiff = PropBBOX - TrapBBOX            
                idxTouch = np.argwhere(np.logical_and(BBOXDiff[0:,0]<0, BBOXDiff[0:,2]>0))

                plt.plot(xTime[idxTouch],Data.at[s, 'Size (mm²)'][idxTouch],'-or',lw=0.5,ms=1.5,figure=fig1,label='Touching UpDown')
                plt.legend(prop={'size': 4})

                fig1.savefig(P + '\\Figures\\' + s + '_SizeTime.png')    


    fig2 = plt.figure(dpi = 200)
    fig2.suptitle(Title + ' - Norm Area vs. time')
    plt.xlabel('Time (min)')
    plt.ylabel('Size (normalized)')


    ###################### plot common graphs #######################
    
    for s in StackList:
        
                   
        xTime = Data.at[s, 'Time (min)'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        Area = Data.at[s, 'Area'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        
        NormArea = Area/Area[0]
        
        plt.plot(xTime,NormArea,'-o',ms=3,figure=fig2,label=s)
        
    plt.legend(prop={'size': 4})
    fig2.savefig(P + '\\Figures\\NormAreaTime_' + Title + '.png')
    
    
        

### 3.2 Plot evolution of propagule shape in time and mean propagule contour

In [ ]:
def plotShapes(P, StackList,ContourData,GlobalData, figname):
    
    # create save folders
    if not os.path.exists(P + '\\Figures\\XYshape\\'):
            os.mkdir(P + '\\Figures\\XYshape\\') 
            
    if not os.path.exists(P + '\\Figures\\ThetaR\\'):
            os.mkdir(P + '\\Figures\\ThetaR\\') 
            
    
    # Number of image in the shortest and longest videos
    nmin = 1 + int(np.min([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    nmax = 1 + int(np.max([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
        
    # Colors
    colorcycle = [plt.get_cmap('jet')(1. * i/nmin) for i in range(nmin)]
    mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)


    ################ Mean Contours #################
    
    # initialize figure for mean contour
    fig0, ax0 = plt.subplots()
    fig0.set_dpi(250)
    fig0.suptitle('MeanContour in time - ' + figname)
    plt.xlabel('(µm)')
    plt.ylabel('(µm)')
    ax0.plot(0,0,'*r')                               
    ax0.set_aspect('equal', adjustable='box')
    
    print('Plotting mean contours...', end='')
    
    for i in range(nmin):
  
        # Plot shape
        meanX = ContourData.loc[(ContourData.index == 'meanCont') & (ContourData['Img'] == i),'X']  
        meanY = ContourData.loc[(ContourData.index == 'meanCont') & (ContourData['Img'] == i),'Y']
        
        ax0.plot(meanX,meanY,'-', lw=0.5)
            
    # Save fig
    fig0.savefig(P + '\\Figures\\MeanContour_' + figname + '.png')
    plt.figure(fig0.number)
    plt.close(fig0.number)
    
    print('Done and saved.')
        
    ################ Individual Contours #################
                  
    for s in StackList:

        # intialize figures
        fig1, ax1 = plt.subplots(dpi = 250)
        fig1.suptitle(figname + ' - ' + s)
        plt.xlabel('(µm)')
        plt.ylabel('(µm)')
        ax1.plot(0,0,'*r')                               
        ax1.set_aspect('equal', adjustable='box')

        fig2, ax2 = plt.subplots(dpi = 250)
        fig2.suptitle(figname + ' - ' + s)
        plt.xlabel('Angle (°)')
        plt.ylabel('Radius (µm)')

        n = int(1 + np.max(ContourData.loc[s, 'Img']))
        
        colorcycle = [plt.get_cmap('jet')(1. * i/n) for i in range(n)]
        mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)
            
        print('Plotting individual contours for gemma : ' + s + '...', end='')
        for i in range(n):

                if (GlobalData.loc[s]['GoodFocus'].to_numpy()[i] == True):

                    A = ContourData[ContourData['Img']==i].loc[s,'Theta']
                    R = ContourData[ContourData['Img']==i].loc[s,'alignR']    

                    X,Y = vf.ToCart(A,R, angle = 'deg') 

                    ax1.plot(X,Y,'-', ms = 0.1, lw = 0.5)
                    ax2.plot(A,R,'-', ms = 0.1, lw = 0.5)
                    

        fig1.savefig(P + '\\Figures\\XYshape\\' + s + '_AligShape.png')
        plt.close(fig1.number)
        fig2.savefig(P + '\\Figures\\ThetaR\\' + s + '_AligRadius.png') 
        plt.figure(fig2.number)
        plt.close(fig2.number)
        
        print('Done and saved.')
 

### 3.3 Plot shape characteristics

#### 3.3.1 Plot similarities in time

In [ ]:
def plotShapeSimi(P,StackList,ContourData,GlobalData, figname):
    
    # Number of image in the shortest and longest videos
    nmin = 1 + int(np.min([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    nmax = 1 + int(np.max([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    
    ### Area and selfSimilarity comparison in time
    
    # initialize figure
    fig0, ax0 = plt.subplots()
    fig0.set_dpi(250)
    fig0.suptitle('selfSimi vs. Area - ' + figname)
    plt.xlabel('Area (norm)')
    plt.ylabel('Similarity to self')                
    
    print('Plot self similarity graphs...', end='')
    
    for s in StackList:
        
        xTime = GlobalData.at[s, 'Time (min)'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]/60
        Area = GlobalData.at[s, 'Area'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]        
        NormArea = Area/Area[0]        
        selfSimi = GlobalData.at[s, 'selfSimi'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]

        ax0.plot(NormArea,selfSimi,'k-', lw = 0.5)
        sc = ax0.scatter(NormArea,selfSimi, c= xTime, cmap = 'jet', vmin=0, vmax = 0.5*nmax)
    
    fig0.colorbar(sc, ax = ax0, label = 'Time (h)')
    
    fig0.savefig(P + '\\Figures\\AreaSelfSimi.png')
    
    print('Done and saved.')
    

    ### Area and meanSimilarity comparison in time
    
    # initialize figure
    fig1, ax1 = plt.subplots()
    fig1.set_dpi(250)
    fig1.suptitle('meanSimi vs. Area - ' + figname)
    plt.xlabel('Area (norm)')
    plt.ylabel('Similarity to mean')                
    
    print('Plot mean similarity graphs...', end='')
    
    for s in StackList:
        
        xTime = GlobalData.at[s, 'Time (min)'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]/60
        Area = GlobalData.at[s, 'Area'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]        
        NormArea = Area/Area[0]        
        meanSimi = GlobalData.at[s, 'meanSimi'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]
        
        ax1.plot(NormArea,meanSimi,'k-', lw = 0.5)
        sc = ax1.scatter(NormArea,meanSimi, c= xTime, cmap = 'jet', vmin=0, vmax = 0.5*nmax)
    
    fig1.colorbar(sc, ax = ax1, label = 'Time (h)')
    
    fig1.savefig(P + '\\Figures\\AreaMeanSimi.png')
    
    print('Done and saved.')

#### 3.3.2 Plot Growth quantifications

Growth Gn(Theta,t) = G(n,Th,t)

In [ ]:
def plotGrowth(P,StackList,ContourData,GlobalData,GrowthMat):
    
    ### Individual growth polar plot    
    
    # Specifics of circular histogram
    N = 20 # number of quadrant
    theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False) # Abcisse
    width = (2*np.pi) / N # bar width    
    bottom = 0 # Bottom level
    
    if not os.path.exists(P + '\\Figures\\Growth\\'):
        os.mkdir(P + '\\Figures\\Growth\\') # create hue image folder   
    
    A = ContourData[ContourData['Img']==0].loc[StackList[0],'Theta'].values+180
    
    for s in StackList:    
        
        print('Plot growth for gemma : ' + s + '...', end='')
        
        if not os.path.exists(P + '\\Figures\\Growth\\' + s  + '_Growth\\'):
            os.mkdir(P + '\\Figures\\Growth\\' + s  + '_Growth\\') # create save folder
       
        n = int(1 + np.max(ContourData.loc[s, 'Img']))
        
        for i in range(n):
            
            
            if (GlobalData.loc[s]['GoodFocus'].to_numpy()[i] == True):
                
 
                G = ContourData[ContourData['Img']==i].loc[s,'Growth']

                # initialize figure
                fig1 = plt.figure(dpi = 250)
                fig1.suptitle(s + ' growth - T = ' +str(GlobalData[GlobalData['Img']==i].loc[s,'Time (min)']))
                ax1 = plt.subplot(111, polar=True)
                ax1.set_ylim(ymin = -1.5, ymax = 100)
                ax1.set_yticks([0,25,50,75,100])
                ax1.set_yticklabels(['','','50%','','100%'])
                
                radii = np.empty(len(theta))
                # Compute hist values
                radii[1:] = [G[(A > np.mod(np.rad2deg(th-width/2),360)) & (A < np.mod(np.rad2deg(th+width/2),360))].mean() for th in theta[1:]]
                radii[0] = G[(A > np.mod(np.rad2deg(theta[0]-width/2),360)) | (A < np.mod(np.rad2deg(theta[0]+width/2),360))].mean()
                
                bars = ax1.bar(theta, radii, width=width, bottom=bottom)

                # Use custom colors
                for r, bar in zip(radii, bars):
                    bar.set_facecolor(plt.cm.jet(r / 100.))
                
                fig1.savefig(P + '\\Figures\\Growth\\' + s  + '_Growth\\' + str(i) + '.png') 
                
                plt.close(fig1.number)
                
                del G
                
        print('Done and saved.\n')
    
    print('Plotting average growth...')

    if not os.path.exists(P + '\\Figures\\Growth\\' + 'MeanGrowth\\'):
            os.mkdir(P + '\\Figures\\Growth\\' + 'MeanGrowth\\') # create save folder

    nmax = 1 + int(np.max([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
   

    for i in range(nmax):
        # initialize figure
        fig1 = plt.figure(dpi = 250)
        fig1.suptitle('Meangrowth - T = ' + str(GlobalData[GlobalData['Img']==i].loc[StackList[0],'Time (min)']))
        ax1 = plt.subplot(polar=True)
        ax1.set_ylim(ymin = -1.5, ymax = 100)
        ax1.set_yticks([0,25,50,75,100])
        ax1.set_yticklabels(['','','50%','','100%'])

        
        radii = np.empty(len(theta))
        # Compute hist values
        radii[1:] = [np.nanmean(GrowthMat[:,(A > np.mod(np.rad2deg(th-width/2),360)) &
                           (A < np.mod(np.rad2deg(th+width/2),360)),i]) for th in theta[1:]]
        
        radii[0] = GrowthMat[:,(A > np.mod(np.rad2deg(theta[0]-width/2),360)) |
                             (A < np.mod(np.rad2deg(theta[0]+width/2),360)),i].mean()
        
        bars = ax1.bar(theta, radii, width=width, bottom=bottom)
        
        # Use custom colors
        for r, bar in zip(radii, bars):
            bar.set_facecolor(plt.cm.jet(r / 100.))

        fig1.savefig(P + '\\Figures\\Growth\\' + 'MeanGrowth\\' + str(i) + '.png') 

        plt.close(fig1.number)

    

#### 3.3.3 Plot assymetry

In [ ]:
def plotAssymetry(P, StackList,GlobalData):
    
    colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/len(StackList)) for i in range(len(StackList))]
    mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)
    
   # Assymmetry in time
    fig0, ax0 = plt.subplots(dpi=250)
    fig0.suptitle('Assymmetry in time')     

    # Assymmetry vs area
    fig1, ax1 = plt.subplots(dpi=250)
    fig1.suptitle('Assymmetry vs area')   
    
    for s in StackList:
        
        xTime = GlobalData.loc[s,'Time (min)']
        Assym = GlobalData.loc[s,'Assymmetry']
        Area = GlobalData.loc[s,'Area']
            
        ax0.plot(xTime,Assym,'-o',ms = 3)
        
        ax1.plot(Area,Assym,'-o', ms = 3)
        
    fig0.legend(StackList,prop={'size': 5})
    fig1.legend(StackList,prop={'size': 5})
    
    fig0.savefig(P + '\\Figures\\AssymTime.png') 
    fig1.savefig(P + '\\Figures\\AssymArea.png') 

    

## 4. Enter data and run

### 4.1 Data details

In [ ]:
# Paths to data
P1 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.18'

P2 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.19'


# List of stacks to analyse

# experiments of 18-03-2021
StackList210318 = ['PPG1','PPG2','PPG3','PPG4','PPG6','PPG7','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']

            # No Rhizoid
StackList210318 = ['PPG2','PPG3','PPG6','PPG7','PPG9','PPG10','PPG11',
                   'PPG14','PPG15','PPG16','PPG18','PPG19',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']      

#StackList210318 = ['PPG2'] # Tests



# experiments of 19-03-2021 (68h, cropped when air arrives, at least 52 images)
StackList210319 = ['PPG1','PPG2','PPG6','PPG8','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22']



#StackList210319 = ['PPG17','PPG1'] # Tests


### 4.2 Loading data 

In [ ]:
## Expe 210318
TimeContourData210318 = pd.read_csv(P1 + '\\ContourData210318.csv', index_col = 'Ind')
GlobalData210318 = pd.read_csv(P1 + '\\GlobalData210318.csv', index_col = 'Ind')    

## Expe 210319
TimeContourData210319 = pd.read_csv(P2 + '\\ContourData210319.csv', index_col = 'Ind')
GlobalData210319 = pd.read_csv(P2 + '\\GlobalData210319.csv', index_col = 'Ind')

### 4.3 Compute contour characteristics

#### 4.3.1 Mean contours

In [ ]:
## Expe 210318
print('\033[1m' + '\033[4m' + '\nAnalyzing experiments from 2021-03-18 :\n' + '\033[0m')
TimeContourData210318 = computeMeanContour(StackList210318,TimeContourData210318,GlobalData210318)

## Expe 210319
print('\033[1m' + '\033[4m' + 'Analyzing experiments from 2021-03-19 :\n' + '\033[0m')
TimeContourData210319 = computeMeanContour(StackList210319,TimeContourData210319,GlobalData210319)


#### 4.3.2 Compute similarities and growth

In [ ]:
## Expe 210318
print('\033[1m' + '\033[4m' + '\nAnalyzing experiments from 2021-03-18 :\n' + '\033[0m')
TimeContourData210318,GlobalData210318,GrowthMat210318 = computeGrowthAndSimi(StackList210318,TimeContourData210318,GlobalData210318)

## Expe 210319
print('\033[1m' + '\033[4m' + 'Analyzing experiments from 2021-03-19 :\n' + '\033[0m')
TimeContourData210319,GlobalData210319,GrowthMat210319 = computeGrowthAndSimi(StackList210319,TimeContourData210319,GlobalData210319)

#### 4.3.3 Compute assymetry

In [ ]:
TimeContourData210318,GlobalData210318 = computeSym(StackList210318,TimeContourData210318,GlobalData210318)

TimeContourData210319,GlobalData210319 = computeSym(StackList210319,TimeContourData210319,GlobalData210319)

### 4.4 Plot results

In [ ]:
print('\033[1m' + '\033[4m' + 'Plotting for experiments from 2021-03-18 :\n' + '\033[0m')

plotArea(StackList210318, GlobalData210318, P1, 'Expe_210318')

plotShapes(P1, StackList210318,TimeContourData210318,GlobalData210318, 'Expe_210318')

plotShapeSimi(P1, StackList210318,TimeContourData210318,GlobalData210318, 'Expe_210318')

plotGrowth(P1,StackList210318,TimeContourData210318,GlobalData210318,GrowthMat210318)

plotAssymetry(P1, StackList210318,GlobalData210318)

print('\n\n')

print('\033[1m' + '\033[4m' + 'Plotting for experiments from 2021-03-19 :\n' + '\033[0m')

plotArea(StackList210319, GlobalData210319, P2, 'Expe_210319')

plotShapes(P2, StackList210319,TimeContourData210319,GlobalData210319, 'Expe_210319')

plotShapeSimi(P2, StackList210319,TimeContourData210319,GlobalData210319, 'Expe_210319')

plotGrowth(P2,StackList210319,TimeContourData210319,GlobalData210319,GrowthMat210319)

plotAssymetry(P2, StackList210319,GlobalData210319)

print('\n\n')

# Test Zone